In [19]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import CountVectorizer

# Datos

In [20]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

train_df.head()

,movie_name,genre,description
0,Silent Hill,"Horror, Mystery","Rose, a desperate mother takes her adopted dau..."
1,Breaking the Waves,"Drama, Romance","In a small and conservative Scottish village, ..."
2,Wind Chill,"Drama, Horror, Thriller",Two college students share a ride home for the...
3,Godmothered,"Family, Fantasy, Comedy",A young and unskilled fairy godmother that ven...
4,Donkey Skin,"Fantasy, Comedy, Music, Romance",A fairy godmother helps a princess disguise he...


# Etiquetas

In [21]:
train_df["genre_list"] = train_df["genre"].apply(lambda x: x.split(", "))

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(train_df["genre_list"])

mlb.classes_

array(['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Drama',
       'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery',
       'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War',
       'Western'], dtype=object)

# MODELO 1: NAIVE BAYES

In [22]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [23]:
vectorizer = CountVectorizer(max_features=5000, stop_words='english')

X = vectorizer.fit_transform(train_df["description"])

X.shape

(8475, 5000)

In [24]:
model = OneVsRestClassifier(MultinomialNB())
model.fit(X, y)
y_pred = model.predict(X)

print(classification_report(y, y_pred, target_names=mlb.classes_))

                 precision    recall  f1-score   support

         Action       0.69      0.84      0.76      1974
      Adventure       0.64      0.79      0.71      1429
      Animation       0.63      0.82      0.71       805
         Comedy       0.76      0.79      0.77      3046
          Crime       0.68      0.83      0.75      1362
          Drama       0.80      0.82      0.81      3957
         Family       0.73      0.85      0.78       969
        Fantasy       0.63      0.77      0.69       962
        History       0.54      0.82      0.65       442
         Horror       0.71      0.85      0.78      1159
          Music       0.69      0.78      0.73       250
        Mystery       0.61      0.77      0.68       813
        Romance       0.64      0.82      0.72      1483
Science Fiction       0.68      0.83      0.75      1004
       TV Movie       0.67      0.50      0.57       101
       Thriller       0.71      0.81      0.75      2289
            War       0.57    

c:\Users\akaca\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [25]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print(classification_report(y_val, y_pred, target_names=mlb.classes_))

                 precision    recall  f1-score   support

         Action       0.64      0.75      0.69       411
      Adventure       0.51      0.62      0.56       278
      Animation       0.50      0.56      0.53       172
         Comedy       0.67      0.68      0.67       625
          Crime       0.57      0.65      0.61       273
          Drama       0.65      0.68      0.67       789
         Family       0.58      0.61      0.60       178
        Fantasy       0.41      0.45      0.43       189
        History       0.34      0.42      0.37        90
         Horror       0.58      0.60      0.59       231
          Music       0.45      0.43      0.44        53
        Mystery       0.42      0.44      0.43       163
        Romance       0.46      0.67      0.54       268
Science Fiction       0.57      0.70      0.63       213
       TV Movie       0.00      0.00      0.00        17
       Thriller       0.62      0.69      0.65       471
            War       0.44    

c:\Users\akaca\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# MODELO 2: LOGISTIC REGRESSION

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [29]:
# Vectorización TF-IDF
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    stop_words='english',
    min_df=2
)
X_tfidf = tfidf.fit_transform(train_df["description"])


X_train, X_val, y_train, y_val = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)
model_lr = OneVsRestClassifier(LogisticRegression(max_iter=1000))
model_lr.fit(X_train, y_train)
y_prob = model_lr.predict_proba(X_val)

threshold = 0.3
y_pred = (y_prob >= threshold).astype(int)
print(classification_report(y_val, y_pred, target_names=mlb.classes_))

                 precision    recall  f1-score   support

         Action       0.67      0.71      0.69       411
      Adventure       0.61      0.46      0.52       278
      Animation       0.81      0.17      0.29       172
         Comedy       0.55      0.86      0.67       625
          Crime       0.72      0.50      0.59       273
          Drama       0.54      0.95      0.69       789
         Family       0.81      0.29      0.43       178
        Fantasy       0.60      0.16      0.26       189
        History       0.40      0.02      0.04        90
         Horror       0.74      0.39      0.51       231
          Music       1.00      0.04      0.07        53
        Mystery       0.62      0.12      0.21       163
        Romance       0.60      0.49      0.54       268
Science Fiction       0.85      0.43      0.57       213
       TV Movie       0.00      0.00      0.00        17
       Thriller       0.60      0.71      0.65       471
            War       0.89    

c:\Users\akaca\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\akaca\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# MODELO 3: TRANSFORMER

In [38]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch
import numpy as np

In [ ]:
train_small = train_df.sample(2000, random_state=42)

train_small["genre_list"] = train_small["genre"].apply(lambda x: x.split(", "))
y_small = mlb.transform(train_small["genre_list"])

In [40]:
dataset = Dataset.from_pandas(train_small)

dataset = dataset.add_column("labels", y_small.astype(float).tolist())

In [41]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(
        example["description"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [42]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [43]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,  # IMPORTANTE: bajo
    per_device_train_batch_size=4,
    logging_steps=10,
    save_strategy="no"
)

In [44]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

c:\Users\akaca\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.603914
20,0.451088
30,0.402391
40,0.385636
50,0.363389
60,0.377003
70,0.411681
80,0.343583
90,0.375714
100,0.386250


TrainOutput(global_step=500, training_loss=0.33620081758499143, metrics={'train_runtime': 3084.3534, 'train_samples_per_second': 0.648, 'train_steps_per_second': 0.162, 'total_flos': 131574426624000.0, 'train_loss': 0.33620081758499143, 'epoch': 1.0})

In [45]:
test_dataset = Dataset.from_pandas(test_df)

test_dataset = test_dataset.map(tokenize, batched=True)

test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

Map:   0%|          | 0/942 [00:00<?, ? examples/s]

# Resultados

## Modelo 1

In [27]:
X_test = vectorizer.transform(test_df["description"])

y_test_pred = model.predict(X_test)

predicted_genres = mlb.inverse_transform(y_test_pred)

test_df["genre"] = [", ".join(genres) for genres in predicted_genres]

output = test_df[["movie_name", "description", "genre"]]

output.to_csv("dataset_test_preds_nb.csv", index=False)

print("✅ Archivo NB generado")

✅ Archivo NB generado


## Modelo 2

In [30]:
X_test = tfidf.transform(test_df["description"])

y_test_prob = model_lr.predict_proba(X_test)

threshold = 0.3
y_test_pred = (y_test_prob >= threshold).astype(int)

predicted_genres = mlb.inverse_transform(y_test_pred)

test_df["genre"] = [", ".join(genres) for genres in predicted_genres]

output = test_df[["movie_name", "description", "genre"]]

output.to_csv("dataset_test_preds_lr.csv", index=False)

print("✅ Archivo LOGISTIC generado")

✅ Archivo LOGISTIC generado


## Modelo 3

In [46]:
predictions = trainer.predict(test_dataset)

logits = predictions.predictions

probs = torch.sigmoid(torch.tensor(logits)).numpy()

threshold = 0.3
y_test_pred = (probs >= threshold).astype(int)

predicted_genres = mlb.inverse_transform(y_test_pred)

test_df["genre"] = [", ".join(genres) for genres in predicted_genres]

output = test_df[["movie_name", "description", "genre"]]

output.to_csv("dataset_test_preds_trf.csv", index=False)

print("✅ Archivo TRANSFORMER generado")

✅ Archivo TRANSFORMER generado
